In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc

In [2]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [3]:
from cuml.preprocessing import MaxAbsScaler
from cuml.preprocessing import MinMaxScaler
from cuml.preprocessing import Normalizer
from cuml.preprocessing import RobustScaler
from cuml.preprocessing import StandardScaler
from cuml.preprocessing import Binarizer
from cuml.preprocessing import FunctionTransformer
from cuml.preprocessing import KBinsDiscretizer
import time

In [4]:
class Normalization(object):
    def __init__(self, dataset):
        global df_global
        global df_label_global
        self.dataset = dataset.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X = cp.array(self.dataset_numpy_array)
        self.df_norm = self.dataset[['HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies',
            'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 
            'Sex', 'Age', 'Education','Income']]
        self.df_norm_numpy_array = self.df_norm.copy().to_numpy()
        self.dataset_label = self.dataset[['HeartDiseaseorAttack']]
        self.df_global = self.df_norm
        self.df_label_global = self.dataset_label


    def MaxAbsScaler(self):
        global maxAbsScaler_global

        transformer = MaxAbsScaler().fit(self.df_norm_numpy_array)
        df_maxAbs_scaler_transform = transformer.transform(self.df_norm_numpy_array)
        
        maxAbsScaler_global = cudf.DataFrame(df_maxAbs_scaler_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])
        
    def MinMaxScaler(self):
        global MinMaxScaler_global

        transformer = MinMaxScaler().fit(self.df_norm_numpy_array)
        df_MinMaxScaler_transform = transformer.transform(self.df_norm_numpy_array)

        MinMaxScaler_global = cudf.DataFrame(df_MinMaxScaler_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def Normalizer(self):
        global normalizer_global

        transformer = Normalizer().fit(self.df_norm_numpy_array)
        df_normalizer_transform = transformer.transform(self.df_norm_numpy_array)

        normalizer_global = cudf.DataFrame(df_normalizer_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def RobustScaler(self):
        global robust_scaler_global

        transformer = RobustScaler().fit(self.df_norm_numpy_array)
        df_robust_scaler_transform = transformer.transform(self.df_norm_numpy_array)

        robust_scaler_global = cudf.DataFrame(df_robust_scaler_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def StandardScaler(self):
        global standard_scaler_global

        transformer = StandardScaler().fit(self.df_norm_numpy_array)
        df_standard_scaler_transform = transformer.transform(self.df_norm_numpy_array)

        standard_scaler_global = cudf.DataFrame(df_standard_scaler_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def Binarizer(self):
        global binarizer_global

        transformer = Binarizer().fit(self.df_norm_numpy_array)
        df_binarizer_transform = transformer.transform(self.df_norm_numpy_array)

        binarizer_global = cudf.DataFrame(df_binarizer_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def FunctionTransformer(self):
        global function_transformer_global

        transformer = FunctionTransformer(func=cp.log1p)
        df_function_transformer_transform = transformer.transform(self.df_norm_numpy_array)

        function_transformer_global = cudf.DataFrame(df_function_transformer_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def KBinsDiscretizer(self):
        global KBinsDiscretizer_global

        transformer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform').fit(self.df_norm_numpy_array)
        df_KBinsDiscretizer_transform = transformer.transform(self.df_norm_numpy_array)

        KBinsDiscretizer_global = cudf.DataFrame(df_KBinsDiscretizer_transform, columns = ['HighBP', 'HighChol', 'CholCheck', 
                                                                                    'BMI','Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 
                                                                                    'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 
                                                                                    'GenHlth','MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 
                                                                                    'Age', 'Education','Income'])

    def df_merge(self):
        global maxAbsScaler_merged_global
        global MinMaxScaler_merged_global
        global normalizer_merged_global
        global robust_scaler_merged_global
        global standard_scaler_merged_global
        global binarizer_merged_global
        global function_transformer_merged_global
        global KBinsDiscretizer_merged_global

        maxAbsScaler_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        MinMaxScaler_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        normalizer_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        robust_scaler_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        standard_scaler_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        binarizer_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        function_transformer_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        KBinsDiscretizer_merged_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)

        


    def main(self):
        st = time.time()
        self.MaxAbsScaler()
        self.MinMaxScaler()
        self.Normalizer()
        self.RobustScaler()
        self.StandardScaler()
        self.Binarizer()
        self.FunctionTransformer()
        self.KBinsDiscretizer()
        self.df_merge()

        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [5]:
norm = Normalization(df)

In [6]:
norm.main()

Execution time: 0.6911847591400146 seconds


In [7]:
class Data_Generation(object):

    def __init__(self, dataset, size, data_type):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.size = size
        self.data_type = data_type
        self.HighBP_nums = []
        self.HighChol_nums = []
        self.CholCheck_nums = []
        self.Smoker_nums = []
        self.Stroke_nums = []
        self.PhysActivity_nums = []
        self.Fruits_nums = []
        self.Veggies_nums= []
        self.HvyAlcoholConsump_nums = []
        self.AnyHealthcare_nums = []
        self.NoDocbcCost_nums = []
        self.DiffWalk_nums = []
        self.Sex_nums = []
        self.BMI_nums = []
        self.Diabetes_nums = []
        self.GenHlth_nums = []
        self.MentHlth_nums = []
        self.PhysHlth_nums = []
        self.Age_nums = []
        self.Education_nums = []
        self.Income_nums = []
        self.generated_data = []

    def generating_data(self):
        self.HighBP_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.HighChol_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.CholCheck_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Smoker_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Stroke_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.PhysActivity_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Fruits_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Veggies_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.HvyAlcoholConsump_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.AnyHealthcare_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.NoDocbcCost_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.DiffWalk_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Sex_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.BMI_nums = cp.random.choice(range(12, 99), size=self.size).astype(self.data_type)
        self.Diabetes_nums = cp.random.choice(range(0, 3), size = self.size).astype(self.data_type)
        self.GenHlth_nums = cp.random.choice(range(1, 6), size = self.size).astype(self.data_type)
        self.MentHlth_nums = cp.random.choice(range(0, 31), size = self.size).astype(self.data_type)
        self.PhysHlth_nums = cp.random.choice(range(0, 31), size = self.size).astype(self.data_type)
        self.Age_nums = cp.random.choice(range(1, 14), size = self.size).astype(self.data_type)
        self.Education_nums = cp.random.choice(range(1, 7), size = self.size).astype(self.data_type)
        self.Income_nums = cp.random.choice(range(1, 9), size = self.size).astype(self.data_type)
        
        
    def generating_dataframe(self):
        global generated_data_global

        self.generated_data = cudf.DataFrame({"HighBP": self.HighBP_nums, 
                                            'HighChol': self.HighChol_nums,
                                            'CholCheck' : self.CholCheck_nums,
                                            'BMI' : self.BMI_nums, 
                                            'Smoker' : self.Smoker_nums, 
                                            'Stroke' : self.Stroke_nums, 
                                            'Diabetes' : self.Diabetes_nums,
                                            'PhysActivity' : self.PhysActivity_nums, 
                                            'Fruits' : self.Fruits_nums, 
                                            'Veggies' : self.Veggies_nums,
                                            'HvyAlcoholConsump' : self.HvyAlcoholConsump_nums, 
                                            'AnyHealthcare' : self.AnyHealthcare_nums, 
                                            'NoDocbcCost' : self.NoDocbcCost_nums,
                                            'GenHlth' : self.GenHlth_nums, 
                                            'MentHlth' : self.MentHlth_nums, 
                                            'PhysHlth' : self.PhysHlth_nums, 
                                            'DiffWalk' : self.DiffWalk_nums, 
                                            'Sex' : self.Sex_nums, 
                                            'Age' : self.Age_nums, 
                                            'Education' : self.Education_nums,
                                            'Income' : self.Income_nums})

        
        generated_data_global = self.generated_data

    def main(self):
        st = time.time()
        self.generating_data()
        self.generating_dataframe()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [8]:
torch.cuda.empty_cache()
gc.collect()

20

In [9]:
gen_df_1 = Data_Generation(df, 2000000, float)

In [10]:
gen_df_1.main()

Execution time: 0.05022120475769043 seconds


In [11]:
generated_data_global_1 = generated_data_global

In [12]:
from cuml import LinearRegression
from cuml.linear_model import LinearRegression
from cuml.metrics import mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score, accuracy_score, confusion_matrix, kl_divergence
from cuml.metrics import log_loss, roc_auc_score, nan_euclidean_distances, pairwise_distances, sparse_pairwise_distances
from cuml.model_selection import train_test_split, KFold

In [13]:
class machine_learing(object):
    def __init__(self, dataset, generated_data_global_1):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.X = cudf.DataFrame(self.dataset.copy().drop(['HeartDiseaseorAttack'], axis = 1))
        self.y = cudf.DataFrame(self.dataset['HeartDiseaseorAttack'].copy())
        self.generated_data_global_1 = generated_data_global_1.copy().reset_index(drop = True)

    def train_test_split(self):
        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        X_train, X_test, y_train, y_test = train_test_split(self.X, self.y, test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def Linear_Regression(self):
        global lr_global
        global lr_predict_test_global
        global lr_predict_global_1
        global mean_squared_error_global_lr
        global mean_squared_log_error_global_lr
        global median_absolute_error_global_lr
        global r2_score_global_lr
        global accuracy_score_global_lr
        global kl_divergence_global_lr
        global log_loss_global_lr
        global roc_auc_score_global_lr
        global nan_euclidean_distances_global_lr
        global pairwise_distances_global_lr
        global sparse_pairwise_distances_global_lr

        model = LinearRegression(fit_intercept = True, algorithm = "eig")
        lr = model.fit(X_train_global.copy(), y_train_global.copy())
        lr_global = lr
        
        preds_test = model.predict(X_test_global.copy())
        lr_predict_test_global = preds_test
        
        mean_squared_error_global_lr = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_lr = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_lr = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_lr = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_lr = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_lr = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_lr = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_lr = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_lr = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_lr = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_lr = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        lr_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def LogisticRegression(self):
        
        global logr_global
        global logr_predict_test_global
        global logr_predict_global_1
        global mean_squared_error_global_logr
        global mean_squared_log_error_global_logr
        global median_absolute_error_global_logr
        global r2_score_global_logr
        global accuracy_score_global_logr
        global kl_divergence_global_logr
        global log_loss_global_logr
        global roc_auc_score_global_logr
        global nan_euclidean_distances_global_logr
        global pairwise_distances_global_logr
        global sparse_pairwise_distances_global_logr

        model = cuml.LogisticRegression(penalty='l2', tol=0.0001, C=1.0, fit_intercept=True, class_weight=None, max_iter=1000, 
                                        linesearch_max_iter=50, l1_ratio=None, solver='qn', lbfgs_memory=5, penalty_normalized=True, 
                                        verbose=False, handle=None, output_type=None)
        logr = model.fit(X_train_global.copy(), y_train_global.copy().values.ravel())
        logr_global = logr

        preds_test = model.predict(X_test_global.copy())
        logr_predict_test_global = preds_test
        
        mean_squared_error_global_logr = mean_squared_error(y_test_global.copy(), preds_test)
        mean_squared_log_error_global_logr = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_logr = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_logr = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_logr = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_logr = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_logr = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_logr = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_logr = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_logr = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_logr = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        logr_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    
    def RidgeRegression(self):
        global ridr_global
        global ridr_predict_test_global
        global ridr_predict_global_1
        global mean_squared_error_global_ridr
        global mean_squared_log_error_global_ridr
        global median_absolute_error_global_ridr
        global r2_score_global_ridr
        global accuracy_score_global_ridr
        global kl_divergence_global_ridr
        global log_loss_global_ridr
        global roc_auc_score_global_ridr
        global nan_euclidean_distances_global_ridr
        global pairwise_distances_global_ridr
        global sparse_pairwise_distances_global_ridr 

        model = cuml.Ridge(alpha=1e-5, fit_intercept=True, solver='auto', copy_X=True, handle=None, output_type=None, verbose=False)
        ridr = model.fit(X_train_global.copy(), y_train_global.copy().values.ravel())
        ridr_global = ridr

        preds_test = model.predict(X_test_global.copy())
        ridr_predict_test_global = preds_test
        
        mean_squared_error_global_ridr = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_ridr = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_ridr = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_ridr = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_ridr = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_ridr = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_ridr = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_ridr = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_ridr = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_ridr = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_ridr = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        ridr_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def LassoRegression(self):
        global lasr_global
        global lasr_predict_test_global
        global lasr_predict_global_1
        global mean_squared_error_global_lasr
        global mean_squared_log_error_global_lasr
        global median_absolute_error_global_lasr
        global r2_score_global_lasr
        global accuracy_score_global_lasr
        global kl_divergence_global_lasr
        global log_loss_global_lasr
        global roc_auc_score_global_lasr
        global nan_euclidean_distances_global_lasr
        global pairwise_distances_global_lasr
        global sparse_pairwise_distances_global_lasr 

        model = cuml.Lasso(alpha = 0.1, fit_intercept=True, max_iter=1000, tol=0.001, solver='cd', 
                           selection='cyclic', handle=None, output_type=None, verbose=False)
        lasr = model.fit(X_train_global.copy(), y_train_global.copy().values.ravel())
        lasr_global = lasr

        preds_test = model.predict(X_test_global.copy())
        lasr_predict_test_global = preds_test
        
        mean_squared_error_global_lasr = mean_squared_error(y_test_global.copy(), preds_test)
        mean_squared_log_error_global_lasr = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_lasr = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_lasr = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_lasr = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_lasr = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_lasr = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_lasr = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_lasr = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_lasr = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_lasr = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        lasr_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def ElasticNetRegression(self):
        global enr_global
        global enr_predict_test_global
        global enr_predict_global_1
        global mean_squared_error_global_enr
        global mean_squared_log_error_global_enr
        global median_absolute_error_global_enr
        global r2_score_global_enr
        global accuracy_score_global_enr
        global kl_divergence_global_enr
        global log_loss_global_enr
        global roc_auc_score_global_enr
        global nan_euclidean_distances_global_enr
        global pairwise_distances_global_enr
        global sparse_pairwise_distances_global_enr 

        model = cuml.ElasticNet(alpha = 0.1, l1_ratio=0.5, fit_intercept=True, max_iter=1000, tol=0.001, solver='cd', selection='cyclic', 
                                handle=None, output_type=None, verbose=False)
        enr = model.fit(X_train_global.copy(), y_train_global.copy().values.ravel())
        enr_global = enr

        preds_test = model.predict(X_test_global.copy())
        enr_predict_test_global = preds_test
        
        mean_squared_error_global_enr = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_enr = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_enr = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_enr = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_enr = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_enr = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_enr = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_enr = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_enr = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_enr = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_enr = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        enr_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)

    def MiniBatchSGDClassifier(self):
        global mbc_global
        global mbc_predict_test_global
        global mbc_predict_global_1
        global mean_squared_error_global_mbc
        global mean_squared_log_error_global_mbc
        global median_absolute_error_global_mbc
        global r2_score_global_mbc
        global accuracy_score_global_mbc
        global kl_divergence_global_mbc
        global log_loss_global_mbc
        global roc_auc_score_global_mbc
        global nan_euclidean_distances_global_mbc
        global pairwise_distances_global_mbc
        global sparse_pairwise_distances_global_mbc 

        model = cuml.MBSGDClassifier(loss='hinge', penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, epochs=1000, tol=0.001, 
                                     shuffle=True, learning_rate='constant', eta0=0.001, power_t=0.5, batch_size=32, n_iter_no_change=5, 
                                     handle=None, verbose=False, output_type=None)
        mbc = model.fit(X_train_global.copy(), y_train_global.copy().values.ravel())
        mbc_global = mbc

        preds_test = model.predict(X_test_global.copy())
        mbc_predict_test_global = preds_test
        
        mean_squared_error_global_mbc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_mbc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_mbc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_mbc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_mbc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_mbc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_mbc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_mbc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_mbc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_mbc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_mbc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.generated_data_global_1)
        mbc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
    def main(self):
        st = time.time()
        self.train_test_split()
        self.Linear_Regression()
        self.LogisticRegression()
        self.RidgeRegression()
        self.LassoRegression()
        self.ElasticNetRegression()
        self.MiniBatchSGDClassifier()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [14]:
ml = machine_learing(df, generated_data_global_1)

In [15]:
torch.cuda.empty_cache()
gc.collect()

0

In [16]:
ml.main()

Execution time: 319.2128231525421 seconds


In [17]:
lr_global

LinearRegression()

In [18]:
logr_global

LogisticRegression()

In [19]:
ridr_global

Ridge(alpha=1e-05)

In [20]:
lasr_global

Lasso(alpha=0.1)

In [21]:
enr_global

ElasticNet(alpha=0.1)

In [22]:
mbc_global

MBSGDClassifier()

In [23]:
class Metrics(object):
    def LinearRegression(self):
        print(' Linear Regression Coef: ')
        print(lr_global.coef_)
        print('\n')
        print('Linear Regression Intercept: ')
        print(lr_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_lr)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_lr)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_lr)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_lr)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_lr)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_lr)
        print('\n')
        print('Unique Values:')
        print(lr_predict_test_global.unique())

    def LogisticRegression(self):
        print(' Logistic Regression Coef: ')
        print(logr_global.coef_)
        print('\n')
        print('Logistic Regression Intercept: ')
        print(logr_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_logr)
        print('\n')
        print('Mean Squared Log Error: ')
        print(mean_squared_log_error_global_logr)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_logr)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_logr)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_logr)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_logr)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_logr)
        print('\n')
        print('Unique Values:')
        print(logr_predict_test_global.unique())

    def RidgeRegression(self):
        print('Ridge Regression Coef: ')
        print(ridr_global.coef_)
        print('\n')
        print('Ridge Regression Intercept: ')
        print(ridr_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_ridr)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_ridr)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_ridr)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_ridr)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_ridr)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_ridr)
        print('\n')
        print('Unique Values:')
        print(ridr_predict_test_global.unique())

    def LassoRegression(self):
        print(' Lasso Regression Coef: ')
        print(lasr_global.coef_)
        print('\n')
        print('Lasso Regression Intercept: ')
        print(lasr_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_lasr)
        print('\n')
        print('Mean Squared Log Error: ')
        print(mean_squared_log_error_global_lasr)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_lasr)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_lasr)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_lasr)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_lasr)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_lasr)
        print('\n')
        print('Unique Values:')
        print(lasr_predict_test_global.unique())

    def ElasticNetRegression(self):
        print('ElasticNet Regression Coef: ')
        print(enr_global.coef_)
        print('\n')
        print('ElasticNet Regression Intercept: ')
        print(enr_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_enr)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_enr)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_enr)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_enr)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_enr)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_enr)
        print('\n')
        print('Unique Values:')
        print(enr_predict_test_global.unique())

    def MiniBatchSGDClassifier(self):
        print('Mini Batch SGD Classifier Coef: ')
        print(mbc_global.coef_)
        print('\n')
        print('Mini Batch SGD Classifier Intercept: ')
        print(mbc_global.intercept_)
        print('\n')
        print('Mean squared error: ')
        print(mean_squared_error_global_mbc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_mbc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_mbc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_mbc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_mbc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_mbc)
        print('\n')
        print('Unique Values:')
        print(mbc_predict_test_global.unique())

    def main(self):
        self.LinearRegression()
        self.LogisticRegression()
        self.RidgeRegression()
        self.LassoRegression()
        self.ElasticNetRegression()
        self.MiniBatchSGDClassifier()

In [24]:
metrics = Metrics()
metrics.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.001243
4     0.023097
5     0.188192
6     0.024651
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.032995
14   -0.000172
15    0.000954
16    0.049416
17    0.052903
18    0.010917
19    0.001196
20   -0.003696
dtype: float64


Linear Regression Intercept: 
-0.14270589939938821


Mean squared error: 
0.07244965790139617


Median Absolute Error: 
0.08481782247394984


R2 Score: 
0.14910311630139816


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.527224  0.608191  0.486775  0.00152

In [25]:
ml_maxAbsScaler = machine_learing(maxAbsScaler_merged_global, generated_data_global_1)

In [26]:
torch.cuda.empty_cache()
gc.collect()

0

In [27]:
ml_maxAbsScaler.main()

Execution time: 330.34952187538147 seconds


In [28]:
torch.cuda.empty_cache()
gc.collect()

0

In [29]:
metrics_maxAbsScaler = Metrics()
metrics_maxAbsScaler.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.528894  0.608393  0.469686  0.060214

In [30]:
ml_MinMaxScaler = machine_learing(MinMaxScaler_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_MinMaxScaler.main()

Execution time: 347.3110032081604 seconds


In [31]:
metrics_MinMaxScaler = Metrics()
metrics_MinMaxScaler.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2        3         4         5         6   \
0  0.528897  0.608394  0.469632  0.06032  

In [32]:
ml_normalizer = machine_learing(normalizer_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_normalizer.main()

Execution time: 347.8926901817322 seconds


In [33]:
metrics_normalizer = Metrics()
metrics_normalizer.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
        0         1         2         3         4         5         6   \
0  0.52889  0.608391  0.469728  0.060155  

In [34]:
ml_robust_scaler = machine_learing(robust_scaler_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_robust_scaler.main()

Execution time: 340.9483997821808 seconds


In [35]:
metrics_robust_scaler = Metrics()
metrics_robust_scaler.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.528891  0.608391  0.469716  0.060169

In [36]:
ml_standard_scaler = machine_learing(standard_scaler_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_standard_scaler.main()

Execution time: 323.0650658607483 seconds


In [37]:
metrics_standard_scaler = Metrics()
metrics_standard_scaler.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.528895  0.608393  0.469676  0.060234

In [38]:
ml_binarizer = machine_learing(binarizer_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_binarizer.main()

Execution time: 319.47592639923096 seconds


In [39]:
metrics_binarizer = Metrics()
metrics_binarizer.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.528895  0.608393  0.469668  0.060246

In [40]:
ml_function_transformer = machine_learing(function_transformer_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_function_transformer.main()

Execution time: 320.2614018917084 seconds


In [41]:
metrics_function_transformer = Metrics()
metrics_function_transformer.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.528894  0.608393  0.469686  0.060213

In [42]:
ml_KBinsDiscretizer = machine_learing(KBinsDiscretizer_merged_global, generated_data_global_1)
torch.cuda.empty_cache()
gc.collect()
ml_KBinsDiscretizer.main()

Execution time: 319.80576968193054 seconds


In [43]:
metrics_KBinsDiscretizer = Metrics()
metrics_KBinsDiscretizer.main()

 Linear Regression Coef: 
0     0.034075
1     0.038439
2     0.013089
3    -0.121782
4     0.023097
5     0.188192
6     0.049302
7     0.004049
8     0.003699
9     0.003634
10   -0.020996
11    0.006711
12    0.007132
13    0.164973
14   -0.005157
15    0.028615
16    0.049416
17    0.052903
18    0.141917
19    0.007175
20   -0.029569
dtype: float64


Linear Regression Intercept: 
-0.14270589939952721


Mean squared error: 
0.07244965790139588


Median Absolute Error: 
0.0848178224741479


R2 Score: 
0.14910311630140172


Accuracy Score: 
0.0


Kl Divergence: 
nan


ROC AUC Score: 
0.839693546295166


Unique Values:
0        0.037284
1        0.152095
2       -0.065624
3       -0.027555
4        0.200609
           ...   
48601    0.063997
48602    0.146988
48603    0.018342
48604    0.086597
48605    0.194234
Length: 48606, dtype: float64
 Logistic Regression Coef: 
         0         1         2         3         4         5         6   \
0  0.528894  0.608393  0.469687  0.060214